In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "intro-header",
   "metadata": {},
   "source": [
    "# E-Commerce Customer Analytics & Segmentation Portfolio\n",
    "## End-to-End Analytics with SQLite, Advanced SQL, and Python\n",
    "\n",
    "### Project Overview\n",
    "This portfolio project delivers an enterprise-grade customer intelligence pipeline using transactional e-commerce data from a UK-based retailer. Using **SQL (SQLite)** as the primary calculation engine and **Python (Pandas, Matplotlib, Seaborn, Squarify)** for statistical aggregation and visualization, this project implements:\n",
    "\n",
    "1. **Data Ingestion & Anomaly Auditing**\n",
    "2. **Exploratory Data Analysis & Geographic Distribution**\n",
    "3. **RFM (Recency, Frequency, Monetary) Customer Segmentation**\n",
    "4. **Churn Risk & Revenue at Risk Analysis (90-Day Inactivity Threshold)**\n",
    "5. **Monthly Cohort Retention Analysis**\n",
    "6. **Customer Lifetime Value (CLV & AOV Modeling)**\n",
    "7. **Market Basket Analysis (Self-Join Cross-Selling)**"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-1-md",
   "metadata": {},
   "source": [
    "## 1. Database Setup & Data Ingestion\n",
    "We load the raw CSV data into a persistent SQLite database (`retail.db`) to enable direct SQL queries."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-1-ingest",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import sqlite3\n",
    "\n",
    "# Load raw transactional dataset\n",
    "df_raw = pd.read_csv('../data/raw/Online_Retail.csv', encoding='ISO-8859-1')\n",
    "\n",
    "# Format invoice date\n",
    "df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'])\n",
    "\n",
    "# Ingest into SQLite database\n",
    "conn = sqlite3.connect('../data/retail.db')\n",
    "df_raw.to_sql('raw_retail', conn, if_exists='replace', index=False)\n",
    "\n",
    "print(\"Data is ready! Total raw records:\", len(df_raw))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-2-head",
   "metadata": {},
   "outputs": [],
   "source": [
    "df_raw.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-2-md",
   "metadata": {},
   "source": [
    "## 2. Data Cleaning & Feature Engineering\n",
    "We audit missing values and anomalies (refunds/cancellations with negative quantities, non-positive unit prices, and missing customer IDs)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-3-missing",
   "metadata": {},
   "outputs": [],
   "source": [
    "query_missing = \"\"\"\n",
    "-- Audit data anomalies in raw dataset\n",
    "SELECT \n",
    "    COUNT(*) AS total_rows,\n",
    "    COUNT(CASE WHEN CustomerID IS NULL THEN 1 END) AS missing_customer_id,\n",
    "    COUNT(CASE WHEN Quantity <= 0 THEN 1 END) AS non_positive_quantity,\n",
    "    COUNT(CASE WHEN UnitPrice <= 0 THEN 1 END) AS non_positive_price\n",
    "FROM raw_retail;\n",
    "\"\"\"\n",
    "pd.read_sql(query_missing, conn)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-3-md",
   "metadata": {},
   "source": [
    "### Creating Production Clean Table (`clean_retail`)\n",
    "Compute total transaction amount (`TotalAmount = Quantity * UnitPrice`) and flag cancellations (`IsCancelled`)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-4-clean-table",
   "metadata": {},
   "outputs": [],
   "source": [
    "clean_table_query = \"\"\"\n",
    "-- Create persistent clean retail table with feature engineering\n",
    "CREATE TABLE IF NOT EXISTS clean_retail AS\n",
    "SELECT \n",
    "    InvoiceNo,\n",
    "    StockCode,\n",
    "    Description,\n",
    "    Quantity,\n",
    "    InvoiceDate,\n",
    "    UnitPrice,\n",
    "    CustomerID,\n",
    "    Country,\n",
    "    (Quantity * UnitPrice) AS TotalAmount,\n",
    "    CASE \n",
    "        WHEN InvoiceNo LIKE 'C%' OR Quantity < 0 THEN 1 \n",
    "        ELSE 0 \n",
    "    END AS IsCancelled\n",
    "FROM raw_retail\n",
    "WHERE UnitPrice > 0;\n",
    "\"\"\"\n",
    "\n",
    "cursor = conn.cursor()\n",
    "cursor.execute(clean_table_query)\n",
    "conn.commit()\n",
    "print(\"Cleaned 'clean_retail' table created successfully.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-4-md",
   "metadata": {},
   "source": [
    "## 3. High-Level KPIs & Geographic Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-5-kpis",
   "metadata": {},
   "outputs": [],
   "source": [
    "query1 = \"\"\"\n",
    "-- Baseline business KPIs (non-cancelled orders)\n",
    "SELECT \n",
    "    ROUND(SUM(TotalAmount), 2) AS total_revenue,\n",
    "    COUNT(DISTINCT CustomerID) AS unique_customers,\n",
    "    COUNT(DISTINCT InvoiceNo) AS total_orders\n",
    "FROM clean_retail\n",
    "WHERE IsCancelled = 0;\n",
    "\"\"\"\n",
    "pd.read_sql(query1, conn)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-6-top-countries",
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "query2 = \"\"\"\n",
    "-- Top 5 revenue-generating countries\n",
    "SELECT \n",
    "    Country,\n",
    "    ROUND(SUM(TotalAmount), 2) AS total_revenue\n",
    "FROM clean_retail\n",
    "WHERE IsCancelled = 0\n",
    "GROUP BY Country\n",
    "ORDER BY total_revenue DESC\n",
    "LIMIT 5;\n",
    "\"\"\"\n",
    "df_top_countries = pd.read_sql(query2, conn)\n",
    "\n",
    "plt.figure(figsize=(10, 5))\n",
    "sns.barplot(\n",
    "    data=df_top_countries, \n",
    "    x='total_revenue', \n",
    "    y='Country', \n",
    "    palette='viridis',\n",
    "    hue='Country'\n",
    ")\n",
    "plt.title('Top 5 Countries by Revenue', fontsize=14, pad=15, weight='bold')\n",
    "plt.xlabel('Revenue (£)', fontsize=12)\n",
    "plt.ylabel('Country', fontsize=12)\n",
    "plt.savefig('../assets/01_top_countries.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-5-md",
   "metadata": {},
   "source": [
    "## 4. RFM (Recency, Frequency, Monetary) Customer Segmentation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-7-rfm-sql",
   "metadata": {},
   "outputs": [],
   "source": [
    "query3 = \"\"\"\n",
    "-- Calculate RFM metrics at customer level\n",
    "SELECT\n",
    "    CustomerID,\n",
    "    ROUND(\n",
    "        JULIANDAY((\n",
    "            SELECT DATE(MAX(InvoiceDate), '+1 day') \n",
    "            FROM clean_retail \n",
    "            WHERE IsCancelled = 0\n",
    "        )) - JULIANDAY(MAX(InvoiceDate)), 0\n",
    "    ) AS recency,\n",
    "    COUNT(DISTINCT InvoiceNo) AS frequency,\n",
    "    ROUND(SUM(TotalAmount), 2) AS monetary\n",
    "FROM clean_retail\n",
    "WHERE IsCancelled = 0 AND CustomerID IS NOT NULL\n",
    "GROUP BY CustomerID;\n",
    "\"\"\"\n",
    "df_rfm = pd.read_sql_query(query3, conn)\n",
    "df_rfm.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-8-rfm-scoring",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Quantile scoring (1 to 5)\n",
    "df_rfm['R_Score'] = pd.qcut(df_rfm['recency'], q=5, labels=[5, 4, 3, 2, 1])\n",
    "df_rfm['F_Score'] = pd.qcut(df_rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])\n",
    "df_rfm['M_Score'] = pd.qcut(df_rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5])\n",
    "df_rfm['RFM_Score'] = df_rfm['R_Score'].astype(str) + df_rfm['F_Score'].astype(str) + df_rfm['M_Score'].astype(str)\n",
    "\n",
    "# Segmentation mapping logic\n",
    "def assign_segment(row):\n",
    "    r = int(row['R_Score'])\n",
    "    f = int(row['F_Score'])\n",
    "    m = int(row['M_Score'])\n",
    "    \n",
    "    if r >= 4 and f >= 4 and m >= 4:\n",
    "        return 'Champions (VIP)'\n",
    "    elif m == 5 and f <= 3:\n",
    "        return 'Big Spenders'\n",
    "    elif r >= 3 and f >= 4:\n",
    "        return 'Loyal Customers'\n",
    "    elif r >= 4 and f in [2, 3]:\n",
    "        return 'Potential Loyalists'\n",
    "    elif r >= 4 and f == 1:\n",
    "        return 'Recent High-Value' if m >= 3 else 'New Customers'\n",
    "    elif r == 3 and f in [2, 3]:\n",
    "        return 'Need Attention'\n",
    "    elif r <= 2 and (f >= 4 or m >= 4):\n",
    "        return 'Cant Lose Them (High Risk)'\n",
    "    elif r in [1, 2] and f in [2, 3]:\n",
    "        return 'At Risk'\n",
    "    else:\n",
    "        return 'Hibernating / Lost'\n",
    "\n",
    "df_rfm['Segment'] = df_rfm.apply(assign_segment, axis=1)\n",
    "\n",
    "segment_summary = df_rfm.groupby('Segment').agg(\n",
    "    Customer_Count=('CustomerID', 'count'),\n",
    "    Avg_Recency=('recency', 'mean'),\n",
    "    Avg_Frequency=('frequency', 'mean'),\n",
    "    Total_Monetary=('monetary', 'sum'),\n",
    "    Avg_Monetary=('monetary', 'mean')\n",
    ").sort_values(by='Total_Monetary', ascending=False)\n",
    "\n",
    "segment_summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-9-treemap",
   "metadata": {},
   "outputs": [],
   "source": [
    "import squarify\n",
    "\n",
    "plot_data = segment_summary.reset_index()\n",
    "labels = [\n",
    "    f\"{row['Segment']}\\n{int(row['Customer_Count'])} Customers\\n£{row['Total_Monetary']/1e3:.0f}K\" \n",
    "    for _, row in plot_data.iterrows()\n",
    "]\n",
    "\n",
    "plt.figure(figsize=(16, 9))\n",
    "colors = [\n",
    "    '#2E7D32', '#4CAF50', '#D32F2F', '#FF9800', '#E57373', \n",
    "    '#81C784', '#9E9E9E', '#FFB74D', '#64B5F6', '#1976D2'\n",
    "]\n",
    "\n",
    "squarify.plot(\n",
    "    sizes=plot_data['Total_Monetary'], \n",
    "    label=labels, \n",
    "    color=colors, \n",
    "    alpha=0.85,\n",
    "    text_kwargs={'fontsize': 9, 'weight': 'bold', 'color': 'black'}\n",
    ")\n",
    "plt.title('RFM Segments: Revenue Share & Customer Volume', fontsize=16, weight='bold', pad=20)\n",
    "plt.axis('off')\n",
    "plt.savefig('../assets/02_rfm_treemap.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-10-segment-bar",
   "metadata": {},
   "outputs": [],
   "source": [
    "df_bar = segment_summary.reset_index().sort_values(by='Avg_Monetary', ascending=False)\n",
    "\n",
    "plt.figure(figsize=(12, 7))\n",
    "ax = sns.barplot(\n",
    "    data=df_bar,\n",
    "    x='Avg_Monetary',\n",
    "    y='Segment',\n",
    "    palette='Blues_r',\n",
    "    edgecolor='black',\n",
    "    linewidth=0.6,\n",
    "    hue='Segment'\n",
    ")\n",
    "\n",
    "for p, (_, row) in zip(ax.patches, df_bar.iterrows()):\n",
    "    width = p.get_width()\n",
    "    ax.text(\n",
    "        width + 120,\n",
    "        p.get_y() + p.get_height() / 2,\n",
    "        f\"£{row['Avg_Monetary']:,.0f}  (n={int(row['Customer_Count'])})\",\n",
    "        va='center',\n",
    "        ha='left',\n",
    "        fontsize=10,\n",
    "        fontweight='bold',\n",
    "        color='#263238'\n",
    "    )\n",
    "\n",
    "plt.title('Average Customer Spend by Segment (with Segment Size)', fontsize=14, weight='bold', pad=20)\n",
    "plt.xlabel('Average Spend (£)', fontsize=11, labelpad=10)\n",
    "plt.ylabel('Customer Segment', fontsize=11)\n",
    "plt.xlim(0, df_bar['Avg_Monetary'].max() * 1.25)\n",
    "sns.despine(top=True, right=True)\n",
    "plt.grid(axis='x', linestyle='--', alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../assets/03_segment_avg_monetary.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-6-md",
   "metadata": {},
   "source": [
    "## 5. Customer Churn & Revenue at Risk Analysis (90-Day Threshold)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-11-churn-sql",
   "metadata": {},
   "outputs": [],
   "source": [
    "query4 = \"\"\"\n",
    "-- Churn Overview & Revenue at Risk Analysis (90-Day Inactivity Threshold)\n",
    "WITH customer_summary AS (\n",
    "    SELECT\n",
    "        CustomerID,\n",
    "        CAST(\n",
    "            (SELECT JULIANDAY(MAX(InvoiceDate)) FROM clean_retail WHERE IsCancelled = 0) \n",
    "            - JULIANDAY(MAX(InvoiceDate)) \n",
    "            AS INT\n",
    "        ) AS recency_days,\n",
    "        ROUND(SUM(Quantity * UnitPrice), 2) AS total_spend\n",
    "    FROM clean_retail\n",
    "    WHERE IsCancelled = 0 \n",
    "      AND CustomerID IS NOT NULL\n",
    "    GROUP BY CustomerID\n",
    "),\n",
    "churn_flagged AS (\n",
    "    SELECT \n",
    "        CustomerID,\n",
    "        total_spend,\n",
    "        CASE WHEN recency_days > 90 THEN 1 ELSE 0 END AS is_churn\n",
    "    FROM customer_summary\n",
    ")\n",
    "SELECT \n",
    "    COUNT(DISTINCT CustomerID) AS total_customers,\n",
    "    SUM(is_churn) AS churned_customers,\n",
    "    COUNT(DISTINCT CustomerID) - SUM(is_churn) AS active_customers,\n",
    "    ROUND(100.0 * SUM(is_churn) / COUNT(DISTINCT CustomerID), 2) AS churn_rate_pct,\n",
    "    ROUND(SUM(CASE WHEN is_churn = 1 THEN total_spend ELSE 0 END), 2) AS revenue_at_risk,\n",
    "    ROUND(SUM(total_spend), 2) AS total_revenue\n",
    "FROM churn_flagged;\n",
    "\"\"\"\n",
    "df_churn = pd.read_sql_query(query4, conn)\n",
    "df_churn"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-12-churn-viz",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Churn Donut Chart\n",
    "active = df_churn['active_customers'].iloc[0]\n",
    "churned = df_churn['churned_customers'].iloc[0]\n",
    "\n",
    "plt.figure(figsize=(7, 7))\n",
    "plt.pie(\n",
    "    [active, churned], \n",
    "    labels=['Active Customers', 'Churned Customers (90+ Days)'],\n",
    "    colors=['#4CAF50', '#E53935'],\n",
    "    autopct='%1.1f%%',\n",
    "    startangle=140,\n",
    "    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2)\n",
    ")\n",
    "plt.title('Customer Churn Proportion (90-Day Inactivity)', fontsize=14, weight='bold')\n",
    "plt.savefig('../assets/04_churn_donut_chart.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-7-md",
   "metadata": {},
   "source": [
    "## 6. Monthly Cohort Retention Analysis\n",
    "Tracking user retention month-over-month based on their initial transaction cohort."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-13-cohort-sql",
   "metadata": {},
   "outputs": [],
   "source": [
    "cohort_query = \"\"\"\n",
    "-- Monthly Cohort Retention Matrix\n",
    "WITH customer_cohort AS (\n",
    "    SELECT \n",
    "        CustomerID,\n",
    "        STRFTIME('%Y-%m', MIN(InvoiceDate)) AS cohort_month\n",
    "    FROM clean_retail\n",
    "    WHERE IsCancelled = 0 AND CustomerID IS NOT NULL\n",
    "    GROUP BY CustomerID\n",
    "),\n",
    "customer_activities AS (\n",
    "    SELECT \n",
    "        r.CustomerID,\n",
    "        c.cohort_month,\n",
    "        STRFTIME('%Y-%m', r.InvoiceDate) AS activity_month,\n",
    "        (CAST(STRFTIME('%Y', r.InvoiceDate) AS INT) - CAST(SUBSTR(c.cohort_month, 1, 4) AS INT)) * 12 + \n",
    "        (CAST(STRFTIME('%m', r.InvoiceDate) AS INT) - CAST(SUBSTR(c.cohort_month, 6, 2) AS INT)) AS cohort_index\n",
    "    FROM clean_retail r\n",
    "    JOIN customer_cohort c ON r.CustomerID = c.CustomerID\n",
    "    WHERE r.IsCancelled = 0\n",
    "    GROUP BY r.CustomerID, c.cohort_month, activity_month, cohort_index\n",
    ")\n",
    "SELECT \n",
    "    cohort_month,\n",
    "    cohort_index,\n",
    "    COUNT(DISTINCT CustomerID) AS active_customers\n",
    "FROM customer_activities\n",
    "GROUP BY cohort_month, cohort_index\n",
    "ORDER BY cohort_month, cohort_index;\n",
    "\"\"\"\n",
    "df_cohort = pd.read_sql_query(cohort_query, conn)\n",
    "\n",
    "# Pivot table for retention percentage\n",
    "cohort_pivot = df_cohort.pivot(index='cohort_month', columns='cohort_index', values='active_customers')\n",
    "cohort_size = cohort_pivot.iloc[:, 0]\n",
    "retention_matrix = cohort_pivot.divide(cohort_size, axis=0)\n",
    "\n",
    "# Heatmap visualization\n",
    "plt.figure(figsize=(15, 8))\n",
    "sns.heatmap(retention_matrix, annot=True, fmt='.1%', cmap='Blues', vmin=0.0, vmax=0.5)\n",
    "plt.title('Monthly Customer Retention Cohorts', fontsize=14, weight='bold', pad=15)\n",
    "plt.xlabel('Cohort Index (Months After First Purchase)', fontsize=12)\n",
    "plt.ylabel('Cohort Month', fontsize=12)\n",
    "plt.savefig('../assets/05_retention_heatmap.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "step-8-md",
   "metadata": {},
   "source": [
    "## 7. Customer Lifetime Value (CLV) & Market Basket Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-14-clv-viz",
   "metadata": {},
   "outputs": [],
   "source": [
    "# CLV vs AOV Bubble Plot\n",
    "plt.figure(figsize=(12, 7))\n",
    "scatter = plt.scatter(\n",
    "    segment_summary['Avg_Frequency'], \n",
    "    segment_summary['Avg_Monetary'], \n",
    "    s=segment_summary['Customer_Count'] * 2,\n",
    "    alpha=0.6, \n",
    "    c=range(len(segment_summary)), \n",
    "    cmap='viridis',\n",
    "    edgecolors='black'\n",
    ")\n",
    "\n",
    "for i, txt in enumerate(segment_summary.index):\n",
    "    plt.annotate(txt, (segment_summary['Avg_Frequency'].iloc[i], segment_summary['Avg_Monetary'].iloc[i] + 150), fontsize=10, weight='bold')\n",
    "\n",
    "plt.yscale('log')\n",
    "plt.title('Customer Lifetime Value (CLV) Profiling by Segment (Bubble Size = Volume)', fontsize=14, weight='bold', pad=15)\n",
    "plt.xlabel('Average Purchase Frequency (Orders)', fontsize=12)\n",
    "plt.ylabel('Average Monetary Value (£ - Log Scale)', fontsize=12)\n",
    "plt.grid(True, linestyle='--', alpha=0.5)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../assets/06_clv_bubble_plot.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-15-basket-sql",
   "metadata": {},
   "outputs": [],
   "source": [
    "basket_query = \"\"\"\n",
    "-- Market Basket Analysis: Top Frequently Co-Purchased Product Pairs\n",
    "WITH valid_items AS (\n",
    "    SELECT DISTINCT InvoiceNo, Description\n",
    "    FROM clean_retail\n",
    "    WHERE IsCancelled = 0 \n",
    "      AND Description IS NOT NULL \n",
    "      AND Description NOT IN ('POSTAGE', 'DOTCOM POSTAGE', 'MANUAL')\n",
    ")\n",
    "SELECT \n",
    "    t1.Description AS product_a,\n",
    "    t2.Description AS product_b,\n",
    "    COUNT(*) AS times_bought_together\n",
    "FROM valid_items t1\n",
    "JOIN valid_items t2 \n",
    "  ON t1.InvoiceNo = t2.InvoiceNo \n",
    " AND t1.Description < t2.Description\n",
    "GROUP BY product_a, product_b\n",
    "ORDER BY times_bought_together DESC\n",
    "LIMIT 10;\n",
    "\"\"\"\n",
    "df_basket = pd.read_sql_query(basket_query, conn)\n",
    "df_basket['pair'] = df_basket['product_a'] + ' + ' + df_basket['product_b']\n",
    "\n",
    "plt.figure(figsize=(12, 6))\n",
    "sns.barplot(data=df_basket, x='times_bought_together', y='pair', palette='mako', hue='pair')\n",
    "plt.title('Top 10 Co-Purchased Product Pairs (Market Basket Analysis)', fontsize=14, weight='bold', pad=15)\n",
    "plt.xlabel('Co-Purchase Frequency', fontsize=12)\n",
    "plt.ylabel('Product Pairs', fontsize=12)\n",
    "plt.savefig('../assets/07_market_basket_top10.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}